# **SETTING UP SCENIC+ ANALYSIS**

# Load Packages

In [ ]:
packages <- c("Seurat","Signac", "here",
  "cowplot","tidyverse", "patchwork", "ggplot2", 
  "DoubletFinder",
  "clustree", "sceasy",
  "Matrix", "matrixStats",
  "Herper", "reticulate", "ggVennDiagram")

purrr::walk(packages, library, character.only = TRUE) #character.only to treat every element of 'packages' as character strings representing package names. 


# Params

In [ ]:
params <- list(
    sample_dir = "/mnt/data/maithe_2025/Multiome_Mouse_Healthy_2025/seurat/subsetting_epi/results/sobjs/",
    sample_id = "SCENICplus_P1-B1_withoutC18",
    output_dir = "output",
    input_dir = "input",
    res_dir = "/mnt/data/maithe_2025/Multiome_Mouse_Healthy_2025/scenicplus_without_C18"
)

In [ ]:
here()

# Set directories

In [ ]:
here(paste0(params$res_dir))

In [ ]:
# set path to results folder, seurat objects folder and tables folder 
res_dir <- here(paste0(params$res_dir))

input_dir <- here(res_dir, params$input_dir)
output_dir <- here(res_dir, params$output_dir)

# access seurat objects 
sample_dir <- here(paste0(params$sample_dir))

# Read seurat object - Epi Diet

In [ ]:
paste0(sample_dir, "Epi_without_C18_Diet_seurat-wnn-integrated_annotated.RDS")

In [ ]:
# read seurat object - Epi Diet
w1 <- readRDS(paste0(sample_dir, "Epi_without_C18_Diet_seurat-wnn-integrated_annotated.RDS"))

In [ ]:
w1

## **1 - Prepare metadata file**

### 1.2 - Get Cell Metadata

In [ ]:
Idents(w1) <- "Annotated"
head(Idents(w1))

In [ ]:
cell_annotations <- Idents(w1)
head(cell_annotations)

In [ ]:
cell_annotations <- as.data.frame(cell_annotations)
colnames(cell_annotations) <- "Cellannotation"
head(cell_annotations)

In [ ]:
unique(sapply(strsplit(rownames(cell_annotations), "_"), `[`, 1))

In [ ]:
cell_annotations$Sample <- sapply(strsplit(rownames(cell_annotations), "_"), `[`, 1)
cell_annotations$barcode <- sapply(strsplit(rownames(cell_annotations), "_"), `[`, 2)
head(cell_annotations)

### 1.3 - Write metadata info to a tsv file

In [ ]:
write.table(cell_annotations,
            file = file.path(input_dir, paste0("forSCENIC_Epi_without_C18_Diet_seurat-wnn-integrated.tsv")),
            sep = "\t", quote = FALSE,
            row.names = TRUE, col.names = TRUE)

## **2 - Prepare RNA RAW Matrix**

### 2.1 - Check Seurat obj

In [ ]:
w1

In [ ]:
DefaultAssay(w1) <- "SCT"

In [ ]:
head(w1@assays$SCT@counts)

### 2.2 Select only SCT assay, which is all that is needed by SCENICplus (RNA-wise)

In [ ]:

w1 <- readRDS("/mnt/data/maithe_2025/Multiome_Mouse_Healthy_2025/seurat/subsetting_epi/results/sobjs/Epi_without_C18_Diet_seurat-wnn-integrated_annotated.RDS")


DefaultAssay(w1) <- "SCT"
sceasy::convertFormat(w1, 
                      from = "seurat", to = "anndata", 
                      assay = "SCT", main_layer = "counts",
                      outFile = "/mnt/data/maithe/Gastric-Origins-Project/2025_analyses_files/forSCENIC_Epi_without_C18_Diet_seurat-wnn-integrated-SCTAssay_counts.h5ad",
                      drop_single_values = FALSE)


## **3 - Finish preparing files in Python**